# Density
This notebook demonstrates using the density module.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax

from popsim.enums import FuelSpecies, Impurity, SpeciesContainer
from popsim.modules.density import Density
from popsim.simulate import make_time_base, simulate
from popsim.visualize import visualize_time_series

species = [FuelSpecies.Deuterium, FuelSpecies.Tritium, Impurity.Helium, Impurity.Tungsten]
species_container = SpeciesContainer(species=species)

volume = 13.0  # Plasma volume [m^3]

# Set up confinement times
additional_assumptions = {
    "k_fuel": 2.0,  # Particle confinement scalar for fuel species.
    "k_impurity": 10.0,  # Particle confinement scalar for impurity species.
}

tau_E = 1.0  # Energy confinement time [s]
particle_confinement_scalars = {
    k: additional_assumptions["k_fuel"] if k in species_container.fuel_species else additional_assumptions["k_impurity"]
    for k in species_container.species
}

# Set up sources and sinks
sources_and_sinks = {k: {} for k in species}
for k in species:
    sources_and_sinks[k]["fueling19"] = volume

sources_and_sinks[FuelSpecies.Deuterium]["fusion"] = 0.0
sources_and_sinks[FuelSpecies.Tritium]["fusion"] = 0.0
sources_and_sinks[Impurity.Helium]["fusion"] = 0.0

# Set initial state
state = Density.State(vol_avg_ion={FuelSpecies.Deuterium: 1.0, FuelSpecies.Tritium: 0.0, Impurity.Helium: 1.0, Impurity.Tungsten: 0.0})

# Set params
params = Density.Params(
    sources_and_sinks=sources_and_sinks,
    species_confinement_time=jax.tree.map(lambda k: k * tau_E, particle_confinement_scalars),
    volume_dot=0.0,
    volume=13,
)

In [ ]:
time_base = make_time_base(t0=0.0, t1=10.0, dt=0.01)
density_module = Density(config=Density.Config())
print(hasattr(state, "keys"))

sol_xarray = simulate(density_module, time_base, state, params)

print(sol_xarray)
visualize_vars = [k for k in sol_xarray.keys() if k.startswith("state.")]
print(visualize_vars)
visualize_time_series(sol_xarray[visualize_vars])